In [23]:
# Cell 0 — Project Imports
import torch

In [24]:
# Cell 1 — TP, FP, FN 계산
def binary_confusion_counts(
    prediction: torch.Tensor,
    target: torch.Tensor,
    foreground_class: int,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """특정 foreground class의 TP, FP, FN, TN을 계산"""
    
    prediction_is_foreground = prediction == foreground_class
    target_is_foreground = target == foreground_class
    
    true_positive = (
        prediction_is_foreground & target_is_foreground
    ).sum()

    false_positive = (
        prediction_is_foreground & ~target_is_foreground
    ).sum()

    false_negative = (
        ~prediction_is_foreground & target_is_foreground
    ).sum()

    true_negative = (
        ~prediction_is_foreground & ~target_is_foreground
    ).sum()
    
    
    return (
        true_positive,
        false_positive,
        false_negative,
        true_negative,
    )

# EX: 0은 background, 1은 foreground organ
target_mask = torch.tensor(
    [
        [
            [
                [0, 1, 1, 0],
                [0, 1, 0, 0],
            ]
        ]
    ],
    dtype=torch.int64,
)
prediction_mask = torch.tensor(
    [
        [
            [
                [0, 1, 0, 0],
                [1, 1, 0, 0],
            ]
        ]
    ],
    dtype=torch.int64,
)

tp, fp, fn, tn = binary_confusion_counts(
    prediction=prediction_mask,
    target=target_mask,
    foreground_class=1,
)

print("Target:\n", target_mask[0, 0])
print("Prediction:\n", prediction_mask[0, 0])
print("TP:", tp.item())
print("FP:", fp.item())
print("FN:", fn.item())
print("TN:", tn.item())
print("Total voxels:", (tp + fp + fn + tn).item())

Target:
 tensor([[0, 1, 1, 0],
        [0, 1, 0, 0]])
Prediction:
 tensor([[0, 1, 0, 0],
        [1, 1, 0, 0]])
TP: 2
FP: 1
FN: 1
TN: 4
Total voxels: 8


In [25]:
# Cell 2 — Dice와 IoU Scratch 계산
# 거대한 background를 잘 맞힌 것 때문에 
# organ segmentation 점수가 부풀려지는 것을 막기 위해서 TN은 두 식 모두에 들어가지 않음.

# Dice: 2 * TP / (2 * TP + FP + FN)
def dice_from_confusion(
    true_positive: torch.Tensor,
    false_positive: torch.Tensor,
    false_negative: torch.Tensor,
) -> torch.Tensor:
    """TP, FP, FN으로 Dice score를 계산"""
    
    numerator = 2 * true_positive
    denominator = (
        2 * true_positive
        + false_positive
        + false_negative
    )
    
    return numerator / denominator


# Intersection over Union: TP / (TP + FP + FN)
def iou_from_confusion(
    true_positive: torch.Tensor,
    false_positive: torch.Tensor,
    false_negative: torch.Tensor,
) -> torch.Tensor:
    """TP, FP, FN으로 Intersection over Union을 계산"""

    denominator = (
        true_positive
        + false_positive
        + false_negative
    )

    return true_positive / denominator

dice = dice_from_confusion(
    true_positive=tp,
    false_positive=fp,
    false_negative=fn,
)

iou = iou_from_confusion(
    true_positive=tp,
    false_positive=fp,
    false_negative=fn,
)

print("Dice:", dice.item())
print("IoU: ", iou.item())



# [집합 관점]
prediction_foreground = prediction_mask == 1 # 여기서 1은 class ID
target_foreground = target_mask == 1

# |P ∩ T|
intersection = (
    prediction_foreground & target_foreground
).sum()

# |P|
prediction_size = prediction_foreground.sum() 

# |T|
target_size = target_foreground.sum()

# |P U T|
union = (
    prediction_foreground | target_foreground
).sum()

# dice: 2|P ∩ T|/(|P| + |T|)
dice_from_sets = (
    2 * intersection
    / (prediction_size + target_size)
)

# intersection over union: |P ∩ T|/|P U T|
iou_from_sets = intersection / union

print("Intersection:", intersection.item())
print("Prediction size:", prediction_size.item())
print("Target size:", target_size.item())
print("Union:", union.item())
print("Dice from sets:", dice_from_sets.item())
print("IoU from sets: ", iou_from_sets.item())

Dice: 0.6666666865348816
IoU:  0.5
Intersection: 2
Prediction size: 3
Target size: 3
Union: 4
Dice from sets: 0.6666666865348816
IoU from sets:  0.5


In [26]:
# Cell 3 — Overlap Test Cases 검증
def evaluate_binary_overlap(
    prediction: torch.Tensor,
    target: torch.Tensor,
    foreground_class: int,
) -> dict[str, torch.Tensor]:
    """특정 class의 confusion counts, Dice와 IoU를 계산"""
    
    tp, fp, fn, tn = binary_confusion_counts(
        prediction=prediction,
        target=target,
        foreground_class=foreground_class,
    )
    
    # dice: 2 * TP / (2 * TP + FP + FN)
    dice = dice_from_confusion(
        true_positive=tp,
        false_positive=fp,
        false_negative=fn,
    )
    
    # iou: TP / (TP + FP + FN)
    iou = iou_from_confusion(
        true_positive=tp,
        false_positive=fp,
        false_negative=fn,
    )
    
    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "dice": dice,
        "iou": iou,
    }
    

# 정답 foreground는 앞의 두 voxel.
case_target = torch.tensor(
    [[[[1, 1, 0, 0]]]],
    dtype=torch.int64,
)

# Perfect: prediction과 target이 완전히 같음
perfect_prediction = torch.tensor(
    [[[[1, 1, 0, 0]]]],
    dtype=torch.int64,
)

# Disjoint: prediction과 target의 교집합이 없음
disjoint_prediction = torch.tensor(
    [[[[0, 0, 1, 1]]]],
    dtype=torch.int64,
)

# Partial: 한 voxel은 맞고, 하나는 놓치며, 하나는 잘못 추가함
partial_prediction = torch.tensor(
    [[[[1, 0, 1, 0]]]],
    dtype=torch.int64,
)

prediction_cases: dict[str, torch.Tensor] = {
    "perfect": perfect_prediction,
    "disjoint": disjoint_prediction,
    "partial": partial_prediction,
}

for case_name, case_prediction in prediction_cases.items():
    result = evaluate_binary_overlap(
        prediction=case_prediction,
        target=case_target,
        foreground_class=1,
    )

    print(f"\n[{case_name}]")
    print(
        "TP={tp}, FP={fp}, FN={fn}, TN={tn}".format(
            tp=result["tp"].item(),
            fp=result["fp"].item(),
            fn=result["fn"].item(),
            tn=result["tn"].item(),
        )
    )
    print("Dice:", result["dice"].item())
    print("IoU: ", result["iou"].item())
    


[perfect]
TP=2, FP=0, FN=0, TN=2
Dice: 1.0
IoU:  1.0

[disjoint]
TP=0, FP=2, FN=2, TN=0
Dice: 0.0
IoU:  0.0

[partial]
TP=1, FP=1, FN=1, TN=1
Dice: 0.5
IoU:  0.3333333432674408


In [27]:
# Cell 4 — Empty-Mask Policy 검증
def overlap_scores_with_empty_policy(
    prediction: torch.Tensor,
    target: torch.Tensor,
    foreground_class: int,
    empty_empty_score: float,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Empty-mask policy를 적용하여 Dice와 IoU를 계산"""
    
    tp, fp, fn, _ = binary_confusion_counts(
        prediction=prediction,
        target=target,
        foreground_class=foreground_class,
    )
    
    dice_denominator = 2 * tp + fp + fn
    iou_denominator = tp + fp + fn
    
    # Prediction & Target이 모두 비어 있으면 -> 0/0
    # 이때 사용할 점수를 정책에 따라 결정
    if iou_denominator.item() == 0:
        empty_score = torch.tensor(
            empty_empty_score,
            dtype=torch.float32,
            device=prediction.device,
        )
        
        return empty_score, empty_score.clone()
    
    dice = 2 * tp / dice_denominator
    iou = tp / iou_denominator

    return dice, iou



empty_mask = torch.tensor(
    [[[[0, 0, 0, 0]]]],
    dtype=torch.int64,
)

one_voxel_organ = torch.tensor(
    [[[[0, 1, 0, 0]]]],
    dtype=torch.int64,
)

empty_cases: dict[
    str,
    tuple[torch.Tensor, torch.Tensor],
] = {
    # Prediction과 target에 organ이 모두 없다.
    "both_empty": (
        empty_mask,
        empty_mask,
    ),

    # 실제 organ은 없지만 prediction이 false positive를 만들었다.
    "false_positive_only": (
        one_voxel_organ,
        empty_mask,
    ),

    # 실제 organ이 있지만 prediction이 완전히 놓쳤다.
    "missed_organ": (
        empty_mask,
        one_voxel_organ,
    ),
}

for case_name, (
    case_prediction,
    case_target,
) in empty_cases.items():
    dice, iou = overlap_scores_with_empty_policy(
        prediction=case_prediction,
        target=case_target,
        foreground_class=1,
        empty_empty_score=1.0,
    )

    print(f"\n[{case_name}]")
    print("Dice:", dice.item())
    print("IoU: ", iou.item())


[both_empty]
Dice: 1.0
IoU:  1.0

[false_positive_only]
Dice: 0.0
IoU:  0.0

[missed_organ]
Dice: 0.0
IoU:  0.0
